In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

## Load data

In [2]:
df_train = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_train.csv")
df_test = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_test.csv")

In [3]:
columns_remove = [
    'VitaminD',
    'YearStart',
]

In [4]:
df_train = df_train[df_train['milk_consumption']<=3]
df_test = df_test[df_test['milk_consumption']<=3]

In [5]:
df_train.drop(columns=columns_remove, inplace=True)
df_test.drop(columns=columns_remove, inplace=True)

In [6]:
category_columns = [
    'Gender','Race' ,'SmokeFam','label','milk_consumption'
]

In [7]:
df_train.columns

Index(['Gender', 'Age', 'Race', 'familysize', 'PIR', 'BMI',
       'WaistCircumference', 'FastingGlucose', 'ALT', 'AST',
       'AlkalinePhosphotase', 'Triglycerides', 'UricAcid', 'Creatinine',
       'HDLCholesterol', 'LDLCholesterol', 'Hemoglobin', 'Hematocrit',
       'MeanCellVolumn', 'MeanCellHemoglobin', 'RedCellDistributionWidth',
       'PlateletCount', 'MeanPlateletVolume', 'Hba1c', 'SmokeFam',
       'milk_consumption', 'label'],
      dtype='object')

In [ ]:
# unuseful_features = ['familysize','SmokeFam','WaistCircumference','AST','ALT','AlkalinePhosphotase','UricAcid','LDLCholesterol','Hematocrit','MeanCellHemoglobin','PlateletCount', 'MeanPlateletVolume','familysize']

In [8]:
unuseful_features = ['SmokeFam','WaistCircumference','AST','ALT','AlkalinePhosphotase','UricAcid','LDLCholesterol','Hematocrit','MeanCellHemoglobin','PlateletCount', 'MeanPlateletVolume','familysize']

In [9]:
df_train.drop(columns=unuseful_features,inplace=True)
df_test = df_test[df_train.columns]

In [10]:
df_train.columns

Index(['Gender', 'Age', 'Race', 'PIR', 'BMI', 'FastingGlucose',
       'Triglycerides', 'Creatinine', 'HDLCholesterol', 'Hemoglobin',
       'MeanCellVolumn', 'RedCellDistributionWidth', 'Hba1c',
       'milk_consumption', 'label'],
      dtype='object')

In [11]:
df_train['label'].value_counts()

label
0.0    12380
1.0     3829
Name: count, dtype: int64

In [33]:
df_test['label'].value_counts()

label
0.0    3193
1.0    1437
Name: count, dtype: int64

## Training:  SMOTE with SVMSMOTE

In [13]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, SVMSMOTE



X_train_raw = df_train.drop(columns='label')
y_train=df_train['label']
X_test_raw=df_test.drop(columns=['label'])
y_test=df_test['label']

categorical_cols = ['Gender','Race', 'milk_consumption']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col !='label']

# ====== 1) Preprocessor for numerical + categorical columns ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)
classifiers = {
    "Balanced": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42
    ),
    "ShallowFast": RandomForestClassifier(
        n_estimators=50, max_depth=5, max_features="sqrt", random_state=42
    ),
    "Deep": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, random_state=42
    ),
    "Regularized": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=4, max_features=0.7, random_state=42
    ),
    "Conservative": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20,
        min_samples_leaf=10, max_features="log2", random_state=42
    ),
    "RobustSubsample": RandomForestClassifier(#no ok
        n_estimators=150, max_depth=15, min_samples_split=5,
        min_samples_leaf=3, bootstrap=True, max_features="sqrt",
        random_state=42
    ),
    "Lightweight": RandomForestClassifier(
        n_estimators=50, max_depth=8, max_features=0.5, random_state=42
    ),
    "Heavy": RandomForestClassifier(
        n_estimators=1000, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1
    ),
}
# ====== 2) Classifiers ======
#lightGBM: {'classifier__learning_rate': 0.05, 'classifier__max_depth': 6, 'classifier__n_estimators': 120, 'classifier__num_leaves': 31}
classifiers = {
    'LightGBM': LGBMClassifier(n_estimators=120,max_depth= 6,num_leaves=31, learning_rate=0.05, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=80,learning_rate=0.066, random_state=42, verbosity=0),
    'GradiantBoosting':GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42),
    #'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42),
    #'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes':GaussianNB(var_smoothing= 1e-10),
    'SVM': SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)
}

# ====== 3) SVMSMOTE setup ======
svmsmote = SMOTE(
    sampling_strategy='minority',   # or a float (e.g. 0.5) to avoid full balancing
    k_neighbors=20,              # lower than default (5) to avoid crossing decision boundaries
    random_state=42
)

# ====== 4) Wrapper class for imblearn pipelines ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

# # ====== 5) Train and evaluate models ======
# results = []

# for name, clf in classifiers.items():
#     print(f"\n🚀 Training {name} with SVMSMOTE...")
    
#     pipeline = ImbPipeline(steps=[
#         ('preprocessor', preprocessor),
#         ('smote', svmsmote),
#         ('classifier', clf)
#     ])
    
#     wrapped_model = ImblearnWrapper(pipeline)
    
#     try:
#         wrapped_model.fit(X_train_raw, y_train)
#         y_pred = wrapped_model.predict(X_test_raw)
#         y_proba = wrapped_model.predict_proba(X_test_raw)
        
#         if len(np.unique(y_test)) == 2:
#             auc = roc_auc_score(y_test, y_proba[:, 1])
#         else:
#             auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
        
#         precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
#         recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
#         f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
#         accuracy = accuracy_score(y_test, y_pred)
#         p_class, r_class, f1_class, support = precision_recall_fscore_support(
#             y_test, y_pred, average=None, zero_division=0
#         )

#         print("Per-class metrics:")
#         for i in range(len(p_class)):
#             print(f"Class {i}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")
#         results.append({
#             'Model': name,
#             'Precision (Macro)': precision,
#             'Recall (Macro)': recall,
#             'F1 Score (Macro)': f1,
#             'Accuracy': accuracy,
#             'AUC': auc
#         })
        
#         print(f"✅ {name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
        
#     except Exception as e:
#         print(f"❌ Error training {name}: {e}")

# # ====== 6) Save results ======
# results_df = pd.DataFrame(results)
# print("\n📊 FINAL RESULTS SUMMARY:")
# print("="*60)
# print(results_df.to_string(index=False, float_format='%.4f'))

# results_df.to_csv("svmsmote_classifiers_results.csv", index=False)
# print("\n✅ Results exported to: svmsmote_classifiers_results.csv")

# # ====== 7) Find best model ======
# best_idx = results_df['F1 Score (Macro)'].idxmax()
# best_model = results_df.loc[best_idx, 'Model']
# best_f1 = results_df.loc[best_idx, 'F1 Score (Macro)']
# print(f"\n🏆 BEST MODEL: {best_model} (F1 Score: {best_f1:.4f})")
# ====== 5) Train and evaluate models ======
results = []           # global metrics
per_class_results = [] # per-class metrics

for name, clf in classifiers.items():
    print(f"\n🚀 Training {name} with SVMSMOTE...")
    
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', svmsmote),
        ('classifier', clf)
    ])
    
    wrapped_model = ImblearnWrapper(pipeline)
    
    try:
        wrapped_model.fit(X_train_raw, y_train)
        y_pred = wrapped_model.predict(X_test_raw)
        y_proba = wrapped_model.predict_proba(X_test_raw)
        
        # ---- Global metrics ----
        if len(np.unique(y_test)) == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
        
        precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        accuracy = accuracy_score(y_test, y_pred)
        
        # ---- Per-class metrics ----
        p_class, r_class, f1_class, support = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )
        classes = np.unique(y_test)

        for i, cls in enumerate(classes):
            if len(np.unique(y_test)) == 2:
                y_true_bin = (y_test == cls).astype(int)
                auc_class = roc_auc_score(y_true_bin, y_proba[:, i])
            else:
                auc_class = None

            per_class_results.append({
                "Model": name,
                "Class": cls,
                "Precision": p_class[i],
                "Recall": r_class[i],
                "F1": f1_class[i],
                "Support": support[i],
                "AUC": auc_class
            })

        print("Per-class metrics:")
        for i in range(len(p_class)):
            print(f"Class {classes[i]}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")

        results.append({
            'Model': name,
            'Precision (Macro)': precision,
            'Recall (Macro)': recall,
            'F1 Score (Macro)': f1,
            'Accuracy': accuracy,
            'AUC': auc
        })
        
        print(f"✅ {name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
        
    except Exception as e:
        print(f"❌ Error training {name}: {e}")


# ====== 6) Save results ======
results_df = pd.DataFrame(results)
per_class_df = pd.DataFrame(per_class_results)

print("\n📊 FINAL RESULTS SUMMARY:")
print("="*60)
print(results_df.to_string(index=False, float_format='%.4f'))

results_df.to_csv("svmsmote_classifiers_results.csv", index=False)
per_class_df.to_csv("svmsmote_classifiers_perclass.csv", index=False)

print("\n✅ Results exported to: svmsmote_classifiers_results.csv")
print("✅ Per-class results exported to: svmsmote_classifiers_perclass.csv")


# ====== 7) Find best model ======
best_idx = results_df['F1 Score (Macro)'].idxmax()
best_model = results_df.loc[best_idx, 'Model']
best_f1 = results_df.loc[best_idx, 'F1 Score (Macro)']
print(f"\n🏆 BEST MODEL: {best_model} (F1 Score: {best_f1:.4f})")



🚀 Training LightGBM with SVMSMOTE...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002038 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4589
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Per-class metrics:
Class 0.0: P=0.8054, R=0.7260, F1=0.7636, Support=3193
Class 1.0: P=0.5006, R=0.6103, F1=0.5500, Support=1437
✅ LightGBM - Accuracy: 0.6901, F1: 0.6568, AUC: 0.7326

🚀 Training XGBoost with SVMSMOTE...


c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Per-class metrics:
Class 0.0: P=0.8114, R=0.7141, F1=0.7596, Support=3193
Class 1.0: P=0.4984, R=0.6312, F1=0.5570, Support=1437
✅ XGBoost - Accuracy: 0.6883, F1: 0.6583, AUC: 0.7330

🚀 Training GradiantBoosting with SVMSMOTE...
Per-class metrics:
Class 0.0: P=0.8234, R=0.6586, F1=0.7319, Support=3193
Class 1.0: P=0.4750, R=0.6862, F1=0.5613, Support=1437
✅ GradiantBoosting - Accuracy: 0.6672, F1: 0.6466, AUC: 0.7376

🚀 Training RandomForest with SVMSMOTE...
Per-class metrics:
Class 0.0: P=0.7666, R=0.8312, F1=0.7976, Support=3193
Class 1.0: P=0.5385, R=0.4377, F1=0.4829, Support=1437
✅ RandomForest - Accuracy: 0.7091, F1: 0.6403, AUC: 0.7173

🚀 Training GradientBoosting with SVMSMOTE...
Per-class metrics:
Class 0.0: P=0.8213, R=0.6708, F1=0.7385, Support=3193
Class 1.0: P=0.4802, R=0.6757, F1=0.5614, Support=1437
✅ GradientBoosting - Accuracy: 0.6724, F1: 0.6500, AUC: 0.7384

🚀 Training Naive Bayes with SVMSMOTE...
Per-class metrics:
Class 0.0: P=0.8295, R=0.5424, F1=0.6559, Support=3

In [16]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.base import BaseEstimator, ClassifierMixin
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, SVMSMOTE, KMeansSMOTE, ADASYN

# --- Train / test split already prepared ---
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender','Race', 'milk_consumption']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col !='label']

# ====== 1) Preprocessor ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)

# ====== 2) Oversamplers to test ======
oversamplers = {
    "SMOTE": SMOTE(sampling_strategy="minority", k_neighbors=20, random_state=42),
    "BorderlineSMOTE-1": BorderlineSMOTE(sampling_strategy="minority",kind="borderline-1", k_neighbors=20, random_state=42),
    "BorderlineSMOTE-2": BorderlineSMOTE(sampling_strategy="minority",kind="borderline-2", k_neighbors=20, random_state=42),
    "SVMSMOTE": SVMSMOTE(
            random_state=42,
            sampling_strategy='minority',
            k_neighbors=20,
            m_neighbors=40,
            svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
        ),
    "KMeansSMOTE": KMeansSMOTE(sampling_strategy="minority",k_neighbors=20, random_state=42),
    "ADASYN": ADASYN(sampling_strategy="minority",n_neighbors=20, random_state=42)
}

# ====== 3) Wrapper for pipeline ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

# ====== 4) Train and evaluate LightGBM with all oversamplers ======
results = []
per_class_results = []

for smote_name, smote_method in oversamplers.items():
    print(f"\n🚀 Training LightGBM with {smote_name}...")
    
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', smote_method),
        ('classifier', LGBMClassifier(
            n_estimators=120, max_depth=6, num_leaves=31,
            learning_rate=0.05, random_state=42))
    ])
    
    wrapped_model = ImblearnWrapper(pipeline)
    
    try:
        wrapped_model.fit(X_train_raw, y_train)
        y_pred = wrapped_model.predict(X_test_raw)
        y_proba = wrapped_model.predict_proba(X_test_raw)
        
        # ---- Global metrics ----
        if len(np.unique(y_test)) == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
        
        precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        accuracy = accuracy_score(y_test, y_pred)
        
        # ---- Per-class metrics ----
        p_class, r_class, f1_class, support = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )
        classes = np.unique(y_test)

        for i, cls in enumerate(classes):
            if len(np.unique(y_test)) == 2:
                y_true_bin = (y_test == cls).astype(int)
                auc_class = roc_auc_score(y_true_bin, y_proba[:, i])
            else:
                auc_class = None

            per_class_results.append({
                "Oversampler": smote_name,
                "Class": cls,
                "Precision": p_class[i],
                "Recall": r_class[i],
                "F1": f1_class[i],
                "Support": support[i],
                "AUC": auc_class
            })

        print("Per-class metrics:")
        for i in range(len(p_class)):
            print(f"Class {classes[i]}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")

        results.append({
            'Oversampler': smote_name,
            'Precision (Macro)': precision,
            'Recall (Macro)': recall,
            'F1 Score (Macro)': f1,
            'Accuracy': accuracy,
            'AUC': auc
        })
        
        print(f"✅ {smote_name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
        
    except Exception as e:
        print(f"❌ Error training LightGBM with {smote_name}: {e}")

# ====== 5) Save results ======
results_df = pd.DataFrame(results)
per_class_df = pd.DataFrame(per_class_results)

print("\n📊 FINAL RESULTS SUMMARY:")
print("="*60)
print(results_df.to_string(index=False, float_format='%.4f'))

results_df.to_csv("lightgbm_smote_results.csv", index=False)
per_class_df.to_csv("lightgbm_smote_perclass.csv", index=False)

print("\n✅ Results exported to: lightgbm_smote_results.csv")
print("✅ Per-class results exported to: lightgbm_smote_perclass.csv")

# ====== 6) Find best oversampler ======
best_idx = results_df['F1 Score (Macro)'].idxmax()
best_smote = results_df.loc[best_idx, 'Oversampler']
best_f1 = results_df.loc[best_idx, 'F1 Score (Macro)']
print(f"\n🏆 BEST Oversampler for LightGBM: {best_smote} (F1 Score: {best_f1:.4f})")



🚀 Training LightGBM with SMOTE...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002051 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4589
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Per-class metrics:
Class 0.0: P=0.8054, R=0.7260, F1=0.7636, Support=3193
Class 1.0: P=0.5006, R=0.6103, F1=0.5500, Support=1437
✅ SMOTE - Accuracy: 0.6901, F1: 0.6568, AUC: 0.7326

🚀 Training LightGBM with BorderlineSMOTE-1...


c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001972 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4587
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Per-class metrics:
Class 0.0: P=0.8164, R=0.7087, F1=0.7588, Support=3193
Class 1.0: P=0.4995, R=0.6458, F1=0.5

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001904 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4586
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Per-class metrics:
Class 0.0: P=0.8165, R=0.6702, F1=0.7362, Support=3193
Class 1.0: P=0.4759, R=0.6653, F1=0.5548, Support=1437
✅ BorderlineSMOTE-2 - Accuracy: 0.6687, F1: 0.6455, AUC: 0.7308

🚀 Training LightGBM with SVMSMOTE...


c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002161 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4590
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


❌ Error training LightGBM with KMeansSMOTE: No clusters found with sufficient samples of class 1.0. Try lowering the cluster_balance_threshold or increasing the number of clusters.

🚀 Training LightGBM with ADASYN...
[LightGBM] [Info] Number of positive: 12269, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002069 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4585
[LightGBM] [Info] Number of data points in the train set: 24649, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.497748 -> initscore=-0.009007
[LightGBM] [Info] Start training from score -0.009007
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Per-class metrics:
Class 0.0: P=0.8050, R=0.7228, F1=0.7617, Support=3193
Class 1.0: P=0.4980, R=0.6110, F1=0.5487, Support=1437
✅ ADASYN -

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE
from sklearn.ensemble import GradientBoostingClassifier
# ========== 0) Assume dataset split ==========
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender','Race', 'milk_consumption']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col != 'label']

# ====== 1) Preprocessor for numerical + categorical columns ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)
dt_setups = {
    "Shallow": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=10, min_samples_leaf=4, random_state=42),
    "Balanced": DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    "Deep": DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42),
    "Randomized": DecisionTreeClassifier(criterion="log_loss", splitter="random", max_depth=None, min_samples_split=20, min_samples_leaf=6, random_state=42),
    "WideFeatures": DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=30, min_samples_split=5, min_samples_leaf=2, max_features="sqrt", random_state=42)
}



gbc_setups = {
    "Balanced": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "ShallowFast": GradientBoostingClassifier(n_estimators=80, learning_rate=0.2, max_depth=2, subsample=0.8, random_state=42),
    "Regularized": GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, min_samples_split=20, min_samples_leaf=5, random_state=42),
    "Deep": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9, random_state=42),
    "RobustSubsample": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, subsample=0.7, max_features="sqrt", random_state=42),
    "Conservative": GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, subsample=0.8, max_features="log2", random_state=42)
}

rf_setups = {
    "Balanced": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42
    ),
    "ShallowFast": RandomForestClassifier(
        n_estimators=50, max_depth=5, max_features="sqrt", random_state=42
    ),
    "Deep": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, random_state=42
    ),
    "Regularized": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=4, max_features=0.6, random_state=42
    ),
    "Conservative": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20,
        min_samples_leaf=10, max_features="log2", random_state=42
    ),
    "RobustSubsample": RandomForestClassifier(#no ok
        n_estimators=150, max_depth=15, min_samples_split=5,
        min_samples_leaf=3, bootstrap=True, max_features="sqrt",
        random_state=42
    ),
    "Lightweight": RandomForestClassifier(
        n_estimators=50, max_depth=8, max_features=0.5, random_state=42
    ),
    "Heavy": RandomForestClassifier(
        n_estimators=1000, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1
    ),
}

#Best: GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)
base_learners = [
    ('lightgbm', LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
    ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
    #('gb', GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)),
    ('RandomForest', rf_setups['Regularized']),
    # ('dt', dt_setups['Shallow']),  
    # ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
    # ('SVM', SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)),
    ('nb', GaussianNB(var_smoothing= 1e-10)),
]


# ====== 3) Meta-learner ======
# meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner = LogisticRegression(
    max_iter=3000,
    solver='saga',          # saga supports L1, L2, elasticnet
    penalty='elasticnet',   # mix between L1 and L2
    l1_ratio=0.6,           # 0.0 -> pure L2, 1.0 -> pure L1
    C=1.0,                  # regularization strength
    class_weight='balanced',
    random_state=42
)
# ====== 4) Stacking classifier ======
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    stack_method="predict_proba",  # use probabilities from base models
    n_jobs=-1
)

# ====== 5) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='minority',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
)

# ====== 6) Full pipeline with SVMSMOTE + Stacking ======
stacking_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote1', svmsmote),
    ('classifier', stacking_clf)
])


# ====== 7) Train and evaluate ======
print("\n🚀 Training Ensemble Stacking Model with SVMSMOTE...")
stacking_pipeline.fit(X_train_raw, y_train)

y_pred = stacking_pipeline.predict(X_test_raw)
y_proba = stacking_pipeline.predict_proba(X_test_raw)

# Evaluate
if len(np.unique(y_test)) == 2:
    auc = roc_auc_score(y_test, y_proba[:, 1])
else:
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')

precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
accuracy = accuracy_score(y_test, y_pred)
p_class, r_class, f1_class, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, zero_division=0
)
print(f1_w)
print("\n📊 Per-class metrics:")
for i in range(len(p_class)):
    print(f"Class {i}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")

print("\n✅ Ensemble Stacking Results")
print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")



🚀 Training Ensemble Stacking Model with SVMSMOTE...
0.7183369289160212

📊 Per-class metrics:
Class 0: P=0.8022, R=0.7811, F1=0.7915, Support=3193
Class 1: P=0.5404, R=0.5720, F1=0.5558, Support=1437

✅ Ensemble Stacking Results
Accuracy: 0.7162, F1: 0.6736, AUC: 0.7363


c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
